##**※ 2023년~2025년 영화 공휴일 여부에 따른 관객수 노이즈를 통제변수로 활용하기 위한 파이썬 코드**

> **(수정) 1-2주차 공휴일 여부 컬럼만 있는 이유 : 수정 전에는 1-2주 관객수만 수집하고 분석을 진행하려 했으나, 데이터 표본 부족으로 인해 가설검정이 어려울 수 있다는 점을 고려하지 않고, 1-2주 각 영화마다 공휴일 여부에 따른 파생 변수 생성**

> 이후 데이터 표본 수를 더 늘리기 위해 1-2주차 관객수에서 1-4관객수까지 수집하고, 다음 파이썬 코드를 활용하여, 최종 마스터 병합 데이터 생성 코드 작성할 때 활용함.

#  영화 2주차 드롭률 분석 코드 설명
> **프로젝트**: 네이버 검색 행동 유형에 따른 영화 체급별 2주차 관객 드롭률 예측 모델링  
> **파일 입력**: `movie_중형_대형 2023/2024/2025_updated.xlsx`, `movie_중대형_입력템플릿.xlsx`  
> **파일 출력**: `movie_드롭률_분석.xlsx` (5개 시트)


---

##  전체 코드 흐름

```
1. 한국 공휴일 캘린더 정의
       ↓
2. 유틸리티 함수 정의 (날짜 파싱, 주말 산출, 공휴일 계산, 분류)
       ↓
3. 엑셀 파일 4개 로드 → 단일 DataFrame 통합
       ↓
4. 영화별 드롭률 계산 + 공휴일 분류
       ↓
5. 요약 통계 출력
       ↓
6. 엑셀 저장 (5개 시트) + 서식 + 헤더 메모
```

---

## 0. 라이브러리 설치 (코랩 환경)

> 코랩에는 `pandas`, `openpyxl`이 기본 설치되어 있습니다.  
> 별도 설치 없이 바로 사용 가능하며, 아래 셀로 버전만 확인합니다.

```python
# 코랩 전용 - 구글 드라이브 마운트 (파일이 Drive에 있을 경우)
from google.colab import drive
drive.mount('/content/drive')

# 작업 디렉토리 확인
import os
os.getcwd()
```

```

In [ ]:
# ============================================================
# [셀 1] 라이브러리 임포트
# ============================================================
import os
from datetime import date, timedelta, datetime

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.comments import Comment

print("✓ 라이브러리 로드 완료")

✓ 라이브러리 로드 완료


In [ ]:
from google.colab import files

print("▼ 아래 버튼을 클릭해 4개 파일을 모두 선택하세요")
uploaded = files.upload()

# 업로드 결과 확인
print("\n✓ 업로드된 파일:")
for fname in uploaded.keys():
    print(f"  - {fname}")

# 파일 경로 상수 정의 (업로드 시 /content/ 에 저장됨)
BASE_DIR = '/content/'
F_2023   = BASE_DIR + 'movie_중형_대형2023_updated.xlsx'
F_2024   = BASE_DIR + 'movie_중형_대형_2024_updated.xlsx'
F_2025   = BASE_DIR + 'movie_중형_대형2025_updated.xlsx'
F_MID    = BASE_DIR + 'movie_중대형_입력템플릿.xlsx'
OUT_PATH = BASE_DIR + 'movie_드롭률_분석.xlsx'

# 4개 파일 존재 여부 검증
required = [F_2023, F_2024, F_2025, F_MID]
missing  = [f for f in required if not os.path.exists(f)]
if missing:
    print("\n⚠️  아래 파일이 없습니다. 파일명을 확인하고 다시 업로드하세요:")
    for f in missing:
        print(f"  - {os.path.basename(f)}")
else:
    print("\n✓ 4개 파일 모두 확인됨. 다음 셀로 진행하세요.")

▼ 아래 버튼을 클릭해 4개 파일을 모두 선택하세요


Saving movie_중대형_입력템플릿.xlsx to movie_중대형_입력템플릿.xlsx
Saving movie_중형_대형_2024_updated.xlsx to movie_중형_대형_2024_updated (1).xlsx
Saving movie_중형_대형2023_updated.xlsx to movie_중형_대형2023_updated (1).xlsx
Saving movie_중형_대형2025_updated.xlsx to movie_중형_대형2025_updated (1).xlsx

✓ 업로드된 파일:
  - movie_중대형_입력템플릿.xlsx
  - movie_중형_대형_2024_updated (1).xlsx
  - movie_중형_대형2023_updated (1).xlsx
  - movie_중형_대형2025_updated (1).xlsx

✓ 4개 파일 모두 확인됨. 다음 셀로 진행하세요.


## 1. 한국 공휴일 캘린더 정의

> ### 왜 공휴일 데이터가 필요한가?
>
> **영화 2주차 드롭률**에는 공휴일이 결정적인 외생변수로 작용합니다.
>
> - **1주차 주말에 공휴일**이 있으면 → 관객이 평소보다 많아 기저가 높아짐 → 드롭률이 **과대** 계산
> - **2주차 주말에 공휴일**이 있으면 → 관객이 평소보다 많아짐 → 드롭률이 **과소** 계산, 심하면 **역주행**처럼 보임
>
> 이 왜곡을 잡아내기 위해 2023~2026년 공휴일을 `date → 이름` 딕셔너리로 하드코딩합니다.
>
> > **2026년을 포함한 이유**: 2025년 12월 31일 개봉 영화의 `2주차 주말`이  
> > 2026년 1월 7~9일로 넘어가기 때문입니다.
>
> #### 공휴일 부스트 계산 범위 (목~월 5일)
> ```
> [목] [금] [토] [일] [월]
>  ↑              ↑
> 인접(adjacent)  인접
>       └─── core(주말 3일) ───┘
> ```
> - **core**: 금/토/일 중 공휴일 수 (0~3)
> - **adjacent**: 목요일·월요일이 공휴일인 경우 (0~2) → 징검다리 long weekend 효과

In [ ]:
HOLIDAYS = {
    # ---- 2023 ----
    date(2023, 1, 1): '신정',
    date(2023, 1, 21): '설연휴', date(2023, 1, 22): '설날', date(2023, 1, 23): '설연휴',
    date(2023, 1, 24): '대체공휴일(설)',
    date(2023, 3, 1): '삼일절',
    date(2023, 5, 5): '어린이날',
    date(2023, 5, 27): '부처님오신날',
    date(2023, 5, 29): '대체공휴일(부처님)',
    date(2023, 6, 6): '현충일',
    date(2023, 8, 15): '광복절',
    date(2023, 9, 28): '추석연휴', date(2023, 9, 29): '추석', date(2023, 9, 30): '추석연휴',
    date(2023, 10, 2): '임시공휴일(추석)',
    date(2023, 10, 3): '개천절',
    date(2023, 10, 9): '한글날',
    date(2023, 12, 25): '성탄절',
    # ---- 2024 ----
    date(2024, 1, 1): '신정',
    date(2024, 2, 9): '설연휴', date(2024, 2, 10): '설날',
    date(2024, 2, 11): '설연휴', date(2024, 2, 12): '대체공휴일(설)',
    date(2024, 3, 1): '삼일절',
    date(2024, 4, 10): '국회의원선거',
    date(2024, 5, 5): '어린이날',
    date(2024, 5, 6): '대체공휴일(어린이날)',
    date(2024, 5, 15): '부처님오신날',
    date(2024, 6, 6): '현충일',
    date(2024, 8, 15): '광복절',
    date(2024, 9, 16): '추석연휴', date(2024, 9, 17): '추석', date(2024, 9, 18): '추석연휴',
    date(2024, 10, 1): '국군의날(임시)',
    date(2024, 10, 3): '개천절',
    date(2024, 10, 9): '한글날',
    date(2024, 12, 25): '성탄절',
    # ---- 2025 ----
    date(2025, 1, 1): '신정',
    date(2025, 1, 27): '임시공휴일(설)',
    date(2025, 1, 28): '설연휴', date(2025, 1, 29): '설날', date(2025, 1, 30): '설연휴',
    date(2025, 3, 1): '삼일절',
    date(2025, 3, 3): '대체공휴일(삼일절)',
    date(2025, 5, 5): '어린이날/부처님오신날',
    date(2025, 5, 6): '대체공휴일',
    date(2025, 6, 6): '현충일',
    date(2025, 8, 15): '광복절',
    date(2025, 10, 3): '개천절',
    date(2025, 10, 5): '추석연휴', date(2025, 10, 6): '추석', date(2025, 10, 7): '추석연휴',
    date(2025, 10, 8): '대체공휴일(추석)',
    date(2025, 10, 9): '한글날',
    date(2025, 12, 25): '성탄절',
    # ---- 2026 (2025년 말 개봉작의 2주차 처리용) ----
    date(2026, 1, 1): '신정',
    date(2026, 2, 16): '설연휴', date(2026, 2, 17): '설날', date(2026, 2, 18): '설연휴',
    date(2026, 3, 1): '삼일절',
    date(2026, 3, 2): '대체공휴일(삼일절)',
    date(2026, 5, 5): '어린이날',
    date(2026, 5, 24): '부처님오신날',
    date(2026, 5, 25): '대체공휴일(부처님)',
    date(2026, 6, 6): '현충일',
    date(2026, 8, 15): '광복절',
    date(2026, 8, 17): '대체공휴일(광복절)',
    date(2026, 9, 24): '추석연휴', date(2026, 9, 25): '추석', date(2026, 9, 26): '추석연휴',
    date(2026, 10, 3): '개천절',
    date(2026, 10, 9): '한글날',
    date(2026, 12, 25): '성탄절',
}

print(f"✓ 공휴일 캘린더 로드 완료 ({len(HOLIDAYS)}개)")

✓ 공휴일 캘린더 로드 완료 (73개)


## 2. 유틸리티 함수 정의

> 이 섹션에서는 코드 전반에서 반복 사용되는 **6개의 핵심 함수**를 정의합니다.  
> 각 함수는 단일 책임 원칙에 따라 역할이 명확히 분리되어 있습니다.

### 2-1. `parse_date()` — 날짜 형식 통일

> #### 왜 필요한가?
>
> 엑셀 파일에서 날짜 데이터는 다양한 형태로 들어옵니다.
>
> | 원본 값 | 형식 | 설명 |
> |---|---|---|
> | `45630` | `int` | 엑셀 날짜 시리얼 (1899-12-30 기준 경과 일수) |
> | `"2025--03-28"` | `str` | 입력 오타 (하이픈 2개) |
> | `"2024-02-22"` | `str` | 정상 문자열 날짜 |
> | `Timestamp('2023-11-22')` | `pd.Timestamp` | pandas가 자동 파싱한 경우 |
>
> → 모든 케이스를 `date` 객체 하나로 통일합니다.
>
> #### 엑셀 시리얼 변환 예시
> ```
> 45630 → 1899-12-30 + 45630일 = 2024-12-04
> ```

In [ ]:
def parse_date(val):
    """다양한 형식의 날짜 값을 Python date 객체로 통일 변환."""
    if pd.isna(val):
        return None

    if isinstance(val, (int, float)):
        # 엑셀 날짜 시리얼: 1899-12-30을 기준일로 삼는 엑셀 규칙
        return (pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(val))).date()

    if isinstance(val, str):
        v = val.replace('--', '-').strip()   # "2025--03-28" 오타 정제
        try:
            return pd.to_datetime(v).date()
        except Exception:
            return None

    if isinstance(val, pd.Timestamp):
        return val.date()

    if isinstance(val, datetime):
        return val.date()

    try:
        return pd.to_datetime(val).date()   # 최후의 범용 파싱 시도
    except Exception:
        return None


# 동작 확인
print(parse_date(45630))           # 2024-12-04 (엑셀 시리얼)
print(parse_date('2025--03-28'))   # 2025-03-28 (오타 보정)
print(parse_date('2023-11-22'))    # 2023-11-22 (정상)

2024-12-04
2025-03-28
2023-11-22


### 2-2. `get_weekend_dates()` — 1주차·2주차 주말 날짜 산출

> #### 한국 영화 주차 산정 기준
>
> 한국 영화 시장에서는 대부분 **수요일** 개봉합니다.
>
> ```
> 예시: 서울의 봄 (2023-11-22, 수요일 개봉)
>
>   [개봉일]   [1주차 주말]      [2주차 주말]
>    11/22(수)  11/24(금) ~ 11/26(일)  12/01(금) ~ 12/03(일)
>                   ↑ 드롭률 분모             ↑ 드롭률 분자
> ```
>
> #### 핵심 계산 공식
> ```python
> days_until_fri = (4 - release_date.weekday()) % 7
> ```
> - `weekday()`: 월=0, 화=1, 수=2, 목=3, **금=4**, 토=5, 일=6
> - 수요일(2) 개봉: `(4-2) % 7 = 2` → 2일 후가 금요일 ✓
> - 금요일(4) 개봉: `(4-4) % 7 = 0` → 당일이 금요일 ✓
> - 일요일(6) 개봉: `(4-6) % 7 = 5` → 5일 후가 금요일 ✓

In [ ]:
def get_weekend_dates(release_date):
    """개봉일 기준으로 1주차·2주차 주말(금/토/일)의 날짜를 반환."""

    # 개봉일로부터 첫 번째 금요일까지 남은 일수 계산
    days_until_fri = (4 - release_date.weekday()) % 7

    wk1_fri = release_date + timedelta(days=days_until_fri)   # 1주차 금요일

    return {
        'wk1_fri': wk1_fri,
        'wk1_sat': wk1_fri + timedelta(days=1),
        'wk1_sun': wk1_fri + timedelta(days=2),
        'wk2_fri': wk1_fri + timedelta(days=7),   # 정확히 1주 후
        'wk2_sat': wk1_fri + timedelta(days=8),
        'wk2_sun': wk1_fri + timedelta(days=9),
    }


# 동작 확인: 서울의 봄 (2023-11-22, 수요일)
wk = get_weekend_dates(date(2023, 11, 22))
print(f"1주차 주말: {wk['wk1_fri']} ~ {wk['wk1_sun']}")   # 11/24 ~ 11/26
print(f"2주차 주말: {wk['wk2_fri']} ~ {wk['wk2_sun']}")   # 12/01 ~ 12/03

1주차 주말: 2023-11-24 ~ 2023-11-26
2주차 주말: 2023-12-01 ~ 2023-12-03


### 2-3. `count_holidays_in()` / `holiday_names_in()` — 공휴일 집계

> 날짜 리스트를 받아 `HOLIDAYS` 딕셔너리와 대조합니다.
> - `count_holidays_in()`: 공휴일 **개수** 반환
> - `holiday_names_in()`: 공휴일 **이름 목록** 반환 (검증·디버깅용)

In [ ]:
def count_holidays_in(dates):
    """날짜 리스트 중 공휴일 개수를 반환."""
    return sum(1 for d in dates if d in HOLIDAYS)


def holiday_names_in(dates):
    """날짜 리스트 중 공휴일의 '날짜(이름)' 문자열 목록을 반환."""
    return [f"{d.strftime('%m/%d')}({HOLIDAYS[d]})" for d in dates if d in HOLIDAYS]


# 동작 확인: 파묘 2주차 주말 (2024-03-01 삼일절 포함)
test_dates = [date(2024, 3, 1), date(2024, 3, 2), date(2024, 3, 3)]
print(count_holidays_in(test_dates))   # 1
print(holiday_names_in(test_dates))    # ['03/01(삼일절)']

1
['03/01(삼일절)']


### 2-4. `compute_holiday_boost()` — 주말 전체 공휴일 부스트 계산

> #### 왜 목요일과 월요일까지 보는가?
>
> 한국 관객의 극장 방문은 **연휴 길이**에 민감합니다.  
> 예를 들어 목요일이 공휴일이면 `목~일` 4일 연휴가 되어 금요일 관객이 급증합니다.
>
> ```
>   계산 대상 날짜 (총 5일)
>   ┌────┬────┬────┬────┬────┐
>   │ 목  │ 금  │ 토  │ 일  │ 월  │
>   └────┴────┴────┴────┴────┘
>     ↑   └── core (0~3) ──┘   ↑
>   adjacent                adjacent
>   (0~2 합산)
> ```
>
> **total = core + adjacent** 가 해당 주말의 총 공휴일 부스트 지수입니다.

In [ ]:
def compute_holiday_boost(wk_dates):
    """주말(금/토/일) 및 인접 목·월요일 공휴일 부스트를 계산."""
    fri, sat, sun = wk_dates
    thu = fri - timedelta(days=1)   # 전날 목요일
    mon = sun + timedelta(days=1)   # 다음날 월요일

    core_holidays = count_holidays_in([fri, sat, sun])   # 주말 3일 공휴일 수
    adj_holidays  = count_holidays_in([thu, mon])        # 인접 목/월 공휴일 수

    return {
        'core'    : core_holidays,
        'adjacent': adj_holidays,
        'total'   : core_holidays + adj_holidays,          # 총 부스트 지수
        'names'   : holiday_names_in([thu, fri, sat, sun, mon]),
    }


# 동작 확인: 2024년 삼일절 주간 (2/29(목)~3/4(월))
boost = compute_holiday_boost([date(2024, 3, 1), date(2024, 3, 2), date(2024, 3, 3)])
print(f"core: {boost['core']}, adjacent: {boost['adjacent']}, total: {boost['total']}")
print(f"공휴일: {boost['names']}")

core: 1, adjacent: 0, total: 1
공휴일: ['03/01(삼일절)']


### 2-5. `classify_case()` — 역주행 4분류 판정

> #### 분류 판정 로직 (결정 트리)
>
> ```
>                ┌─────────────────────────────┐
>                │       드롭률 >= 0?           │
>                └─────────────────────────────┘
>               YES ↙                          ↘ NO (역주행)
>     ┌──────────────────────┐       ┌──────────────────────┐
>     │  공휴일 차이 < 0?    │       │  공휴일 차이 > 0?    │
>     │ (1주차가 공휴일 多)  │       │ (2주차가 공휴일 多)  │
>     └──────────────────────┘       └──────────────────────┘
>       YES ↙       ↘ NO               YES ↙       ↘ NO
>  정상_드롭_   정상_드롭          공휴일_역주행  순수_역주행
>  1주차부스트                    (외생적)     (입소문·서울의봄)
> ```
>
> **공휴일 차이** = 2주차 부스트 − 1주차 부스트
>
> | 분류 | 드롭률 | 공휴일차이 | 해석 |
> |---|---|---|---|
> | 정상_드롭 | ≥ 0 | ≥ 0 | 일반적 2주차 감소 |
> | 정상_드롭_1주차부스트 | ≥ 0 | < 0 | 1주차 공휴일로 기저가 높아 드롭이 과대 계산 |
> | 공휴일_역주행 | < 0 | > 0 | 2주차 공휴일이 관객 끌어올림 (외생 변수) |
> | **순수_역주행** | **< 0** | **≤ 0** | **입소문 효과** (서울의 봄, 소방관 등) |

In [ ]:
def classify_case(drop_rate, holiday_diff):
    """
    드롭률과 공휴일 차이(2주차-1주차)를 기준으로 4가지 케이스를 분류.

    Parameters
    ----------
    drop_rate    : float  드롭률(%)  양수=감소, 음수=역주행
    holiday_diff : int    2주차 공휴일 부스트 - 1주차 공휴일 부스트
    """
    if drop_rate is None or pd.isna(drop_rate):
        return '데이터_부족'

    if drop_rate >= 0:
        # 정상 드롭 케이스
        return '정상_드롭_1주차부스트' if holiday_diff < 0 else '정상_드롭'
    else:
        # 역주행 케이스
        return '공휴일_역주행' if holiday_diff > 0 else '순수_역주행'


# 동작 확인
print(classify_case(45.6,  0))   # 정상_드롭
print(classify_case(38.2, -1))   # 정상_드롭_1주차부스트
print(classify_case(-13.9, 0))   # 순수_역주행  ← 서울의 봄
print(classify_case(-18.9, 1))   # 공휴일_역주행 ← 파묘 (삼일절)

정상_드롭
정상_드롭_1주차부스트
순수_역주행
공휴일_역주행


### 2-6. `safe_num()` / `classify_tier()` — 보조 함수

> - `safe_num()`: 관객수 셀이 빈 문자열·NaN인 경우 안전하게 `None`으로 반환
> - `classify_tier()`: 누적관객수를 받아 체급 문자열 반환

In [ ]:
def safe_num(v):
    """관객수 셀을 안전하게 float으로 변환. 비어있으면 None."""
    try:
        return float(v) if pd.notna(v) and v != '' else None
    except Exception:
        return None


def classify_tier(a):
    """
    누적관객수 기반 체급 분류.

    - 대형   : 500만 이상
    - 중대형 : 300만 ~ 500만 미만
    - 중형   : 50만 ~ 300만 미만
    - 기타   : 50만 미만
    """
    if pd.isna(a):
        return None
    a = int(a)
    if a >= 5_000_000: return '대형'
    if a >= 3_000_000: return '중대형'
    if a >= 500_000:   return '중형'
    return '기타'


# 동작 확인
print(classify_tier(11_854_779))   # 대형  (서울의 봄)
print(classify_tier(3_438_133))    # 중대형 (노량)
print(classify_tier(879_109))      # 중형  (대도시의 사랑법)

대형
중대형
중형


## 3. 데이터 로드 및 통합

> ### 파일 구조 및 통합 전략
>
> | 파일 | 내용 | 주의사항 |
> |---|---|---|
> | `movie_중형_대형2023_updated.xlsx` | 2023년 중형·대형 영화 | `누적 관객수` (공백 포함) |
> | `movie_중형_대형_2024_updated.xlsx` | 2024년 중형·대형 영화 | 컬럼명이 `개봉 1주차 금요일` (차이 있음) |
> | `movie_중형_대형2025_updated.xlsx` | 2025년 중형·대형 영화 | 경쟁작수가 datetime 포맷으로 깨짐 |
> | `movie_중대형_입력템플릿.xlsx` | 300만~500만 중대형 15편 | 시트명 `movie_중대형` |
>
> #### 컬럼명 표준화 (2024년 파일 전용)
> ```python
> '개봉 1주차 금요일' → '개봉 1주 금요일'   # 나머지 파일 기준으로 통일
> ```
>
> #### 중복 제거 전략
> 동일 영화가 여러 파일에 있는 경우 `keep='last'`로 **중대형 템플릿을 우선**합니다.
> (중대형 템플릿이 가장 나중에 append되므로 last = 템플릿 값)

In [ ]:
# 2024 파일 컬럼명을 다른 파일 기준으로 표준화
col_map_2024 = {
    '개봉 1주차 금요일': '개봉 1주 금요일',
    '개봉 1주차 토요일': '개봉 1주 토요일',
    '개봉 1주차 일요일': '개봉 1주 일요일',
    '개봉 2주차 금요일': '개봉 2주 금요일',
    '개봉 2주차 토요일': '개봉 2주 토요일',
    '개봉 2주차 일요일': '개봉 2주 일요일',
}

all_movies = []

# 연도별 파일 순회
for yr, fpath in [(2023, F_2023), (2024, F_2024), (2025, F_2025)]:
    df = pd.read_excel(fpath)
    df = df.rename(columns={'누적 관객수': '누적관객수', **col_map_2024})
    for _, row in df.iterrows():
        d = parse_date(row['개봉일'])
        if d is None:
            continue
        all_movies.append({
            '연도'          : yr,
            '영화명'        : row['영화명'],
            '개봉일'        : d,
            '장르'          : row.get('장르'),
            '관람등급'      : row.get('관람등급'),
            '총 스크린수'   : row.get('총 스크린수'),
            '개봉 1주 금요일': row.get('개봉 1주 금요일'),
            '개봉 1주 토요일': row.get('개봉 1주 토요일'),
            '개봉 1주 일요일': row.get('개봉 1주 일요일'),
            '개봉 2주 금요일': row.get('개봉 2주 금요일'),
            '개봉 2주 토요일': row.get('개봉 2주 토요일'),
            '개봉 2주 일요일': row.get('개봉 2주 일요일'),
            '누적관객수'    : row['누적관객수'],
            '출처'          : f'{yr}_파일',
        })
    print(f"  {yr}: {len(df)}편 로드")

# 중대형 템플릿 (시트명 지정 필수)
df_mid = pd.read_excel(F_MID, sheet_name='movie_중대형')
for _, row in df_mid.iterrows():
    d = parse_date(row['개봉일'])
    if d is None:
        continue
    all_movies.append({
        '연도'          : row['연도'],
        '영화명'        : row['영화명'],
        '개봉일'        : d,
        '장르'          : row.get('장르'),
        '관람등급'      : row.get('관람등급'),
        '총 스크린수'   : row.get('총 스크린수'),
        '개봉 1주 금요일': row.get('개봉 1주 금요일'),
        '개봉 1주 토요일': row.get('개봉 1주 토요일'),
        '개봉 1주 일요일': row.get('개봉 1주 일요일'),
        '개봉 2주 금요일': row.get('개봉 2주 금요일'),
        '개봉 2주 토요일': row.get('개봉 2주 토요일'),
        '개봉 2주 일요일': row.get('개봉 2주 일요일'),
        '누적관객수'    : row['누적관객수'],
        '출처'          : '중대형_템플릿',
    })
print(f"  중대형: {len(df_mid)}편 로드")

# 단일 DataFrame 통합 + 중복 제거 (중대형 템플릿 우선)
df_all = pd.DataFrame(all_movies)
df_all = df_all.drop_duplicates(subset=['영화명', '개봉일'], keep='last').reset_index(drop=True)

print(f"\n✓ 최종 통합: {len(df_all)}편")

  2023: 38편 로드
  2024: 43편 로드
  2025: 35편 로드
  중대형: 15편 로드

✓ 최종 통합: 131편


In [ ]:
results = []

for _, row in df_all.iterrows():
    rd   = row['개봉일']
    wknd = get_weekend_dates(rd)

    # 1·2주차 주말 관객 합산
    w1 = [safe_num(row[c]) for c in ['개봉 1주 금요일', '개봉 1주 토요일', '개봉 1주 일요일']]
    w2 = [safe_num(row[c]) for c in ['개봉 2주 금요일', '개봉 2주 토요일', '개봉 2주 일요일']]

    if None in w1 or None in w2:
        wk1_total = wk2_total = drop = None
    else:
        wk1_total = sum(w1)
        wk2_total = sum(w2)
        drop = (wk1_total - wk2_total) / wk1_total * 100 if wk1_total > 0 else None

    # 공휴일 부스트 산출
    wk1_boost = compute_holiday_boost([wknd['wk1_fri'], wknd['wk1_sat'], wknd['wk1_sun']])
    wk2_boost = compute_holiday_boost([wknd['wk2_fri'], wknd['wk2_sat'], wknd['wk2_sun']])
    holiday_diff = wk2_boost['total'] - wk1_boost['total']

    results.append({
        '연도'              : row['연도'],
        '영화명'            : row['영화명'],
        '개봉일'            : rd.strftime('%Y-%m-%d'),
        '개봉요일'          : '월화수목금토일'[rd.weekday()],
        '체급'              : classify_tier(row['누적관객수']),
        '누적관객수'        : row['누적관객수'],
        '총 스크린수'       : row['총 스크린수'],
        '1주차_주말_관객'   : wk1_total,
        '2주차_주말_관객'   : wk2_total,
        '드롭률(%)'         : round(drop, 2) if drop is not None else None,
        '1주차_공휴일수_주말': wk1_boost['core'],
        '1주차_공휴일수_목월': wk1_boost['adjacent'],
        '2주차_공휴일수_주말': wk2_boost['core'],
        '2주차_공휴일수_목월': wk2_boost['adjacent'],
        '공휴일_차이(2-1)'  : holiday_diff,
        '분류'              : classify_case(drop, holiday_diff),
        '1주차_공휴일명'    : ', '.join(wk1_boost['names']) if wk1_boost['names'] else '',
        '2주차_공휴일명'    : ', '.join(wk2_boost['names']) if wk2_boost['names'] else '',
        '1주차_주말일자'    : f"{wknd['wk1_fri'].strftime('%m/%d')}~{wknd['wk1_sun'].strftime('%m/%d')}",
        '2주차_주말일자'    : f"{wknd['wk2_fri'].strftime('%m/%d')}~{wknd['wk2_sun'].strftime('%m/%d')}",
    })

res_df = pd.DataFrame(results)
print(f"✓ 드롭률 계산 완료: {len(res_df)}편")

✓ 드롭률 계산 완료: 131편


In [ ]:
print("\n■ 분류별 분포")
print(res_df['분류'].value_counts())

reverse_movies = res_df[res_df['드롭률(%)'] < 0].sort_values('드롭률(%)')
print(f"\n■ 역주행 영화 ({len(reverse_movies)}편)")
print(reverse_movies[['연도', '영화명', '개봉일', '체급', '드롭률(%)', '분류']].to_string())

print("\n■ 체급 × 분류 교차표")
print(pd.crosstab(res_df['체급'], res_df['분류'], margins=True))

print("\n■ 체급별 드롭률 기술통계 (정상_드롭 케이스)")
normal = res_df[res_df['분류'] == '정상_드롭']
print(normal.groupby('체급')['드롭률(%)'].describe().round(2))

# ============================================================
# [셀 8] 엑셀 저장 (5개 시트)
# ============================================================
with pd.ExcelWriter(OUT_PATH, engine='openpyxl') as writer:
    res_df.to_excel(writer, sheet_name='전체_드롭률', index=False)
    reverse_movies.to_excel(writer, sheet_name='역주행_영화', index=False)
    pd.crosstab(res_df['체급'], res_df['분류'], margins=True).to_excel(
        writer, sheet_name='체급x분류_교차표')
    res_df.groupby(['체급', '분류'])['드롭률(%)'].agg(
        ['count', 'mean', 'median', 'std', 'min', 'max']).round(2).to_excel(
        writer, sheet_name='체급별_통계')
    pd.DataFrame({
        '항목': ['드롭률 정의', '', '1주차 주말', '2주차 주말', '', '분류 기준',
                 '정상_드롭', '정상_드롭_1주차부스트', '공휴일_역주행', '순수_역주행', '',
                 '공휴일 부스트', '핵심 공휴일', '인접 공휴일', '', '체급 분류',
                 '중형', '중대형', '대형'],
        '설명': [
            '드롭률(%) = (1주차 주말관객 - 2주차 주말관객) / 1주차 주말관객 × 100', '',
            '개봉일 이후 첫 번째 금/토/일 3일간 관객 합계',
            '1주차 주말 + 7일 후 금/토/일 3일간 관객 합계', '', '',
            '드롭률 ≥ 0 AND 공휴일차이(2주차-1주차) ≥ 0',
            '드롭률 ≥ 0 AND 1주차가 더 많은 공휴일 부스트 (드롭 과대 가능)',
            '드롭률 < 0 AND 2주차가 공휴일 부스트 (외생적 역주행)',
            '드롭률 < 0 AND 공휴일 영향 없음 (서울의 봄 타입, 순수 입소문 역주행)', '', '',
            '주말 3일(금/토/일) 중 공휴일로 지정된 날 수',
            '주말 전날 목요일·다음날 월요일이 공휴일인 경우 (long weekend 효과)', '', '',
            '누적관객수 500,000 ~ 2,999,999',
            '누적관객수 3,000,000 ~ 4,999,999',
            '누적관객수 5,000,000 이상',
        ],
    }).to_excel(writer, sheet_name='방법론', index=False)

print(f"✓ 엑셀 초안 저장 완료: {OUT_PATH}")


■ 분류별 분포
분류
정상_드롭           86
정상_드롭_1주차부스트    26
순수_역주행          13
공휴일_역주행          5
데이터_부족           1
Name: count, dtype: int64

■ 역주행 영화 (18편)
       연도              영화명         개봉일   체급  드롭률(%)       분류
41   2024             건국전쟁  2024-02-01   중형 -419.36  공휴일_역주행
110  2025             신의악단  2025-12-31   중형 -130.94   순수_역주행
95   2025              노이즈  2025-06-25   중형 -120.67   순수_역주행
71   2024               청설  2024-11-06   중형  -89.00   순수_역주행
127  2025   극장판 체인소 맨: 레제편  2025-09-24  중대형  -62.73  공휴일_역주행
111  2025           만약에 우리  2025-12-31   중형  -28.93   순수_역주행
9    2023      슈퍼 마리오 브라더스  2023-04-26   중형  -19.72  공휴일_역주행
77   2024               파묘  2024-02-22   대형  -18.86  공휴일_역주행
35   2023             엘리멘탈  2023-06-14   대형  -18.11   순수_역주행
124  2024              소방관  2024-12-04  중대형  -15.43   순수_역주행
33   2023            서울의 봄  2023-11-22   대형  -13.91   순수_역주행
54   2024  콰이어트 플레이스: 첫째 날  2024-06-26   중형  -11.23   순수_역주행
109  2025           윗집 사람들  2025-12-03   중형   -8.41   순수_

In [ ]:
COLUMN_DESCRIPTIONS = {
    '연도'              : "### 연도\n영화 개봉 연도 (2023~2025)\n\n- 출처 파일 기준 연도",
    '영화명'            : "### 영화명\n영화 제목 (KOBIS 등록명 기준)",
    '개봉일'            : "### 개봉일\n극장 개봉일 (YYYY-MM-DD)\n\n- 엑셀 시리얼/오타 문자열을 표준 날짜로 정제",
    '개봉요일'          : "### 개봉요일\n개봉일의 요일\n\n- 한국 영화는 보통 수요일 개봉\n- 금요일 개봉 시 1주차 주말 산정 기준이 달라짐",
    '체급'              : "### 체급\n누적관객수 기반 영화 규모 분류\n\n- 중형: 50만~300만 미만\n- 중대형: 300만~500만 미만\n- 대형: 500만 이상",
    '누적관객수'        : "### 누적관객수\n전체 상영기간 누적 관객수 (명)\n\n- 체급 분류의 기준 지표",
    '총 스크린수'       : "### 총 스크린수\n개봉 시점 전국 상영 스크린 수\n\n- 배급 규모 및 와이드릴리즈 여부 가늠",
    '1주차_주말_관객'   : "### 1주차 주말 관객\n개봉 1주차 금+토+일 관객 합계\n\n- 드롭률 분모",
    '2주차_주말_관객'   : "### 2주차 주말 관객\n개봉 2주차 금+토+일 관객 합계\n\n- 1주차 주말 + 7일 후 동일 요일",
    '드롭률(%)'         : "### 드롭률 (%)\n핵심 종속변수 (Y)\n\n(1주차주말 - 2주차주말) / 1주차주말 x 100\n\n- 양수: 관객 감소 (정상 드롭)\n- 음수: 관객 증가 (역주행)",
    '1주차_공휴일수_주말': "### 1주차 공휴일수 (주말)\n1주차 주말 3일(금/토/일) 중 공휴일 수\n\n- 범위: 0~3",
    '1주차_공휴일수_목월': "### 1주차 공휴일수 (목/월)\n1주차 주말 전날(목)/다음날(월) 공휴일 수\n\n- long weekend 효과 측정\n- 범위: 0~2",
    '2주차_공휴일수_주말': "### 2주차 공휴일수 (주말)\n2주차 주말 3일(금/토/일) 중 공휴일 수\n\n- 범위: 0~3",
    '2주차_공휴일수_목월': "### 2주차 공휴일수 (목/월)\n2주차 주말 전날(목)/다음날(월) 공휴일 수\n\n- 범위: 0~2",
    '공휴일_차이(2-1)'  : "### 공휴일 차이 (2주차 - 1주차)\n역주행 분류의 핵심 변수\n\n2주차 공휴일부스트 - 1주차 공휴일부스트\n\n- 양수: 2주차 공휴일 수혜 -> 공휴일 역주행 의심\n- 0 이하: 공휴일 영향 없음 -> 순수 역주행 판정",
    '분류'              : "### 분류\n드롭률 + 공휴일차이 기반 4분류\n\n- 정상_드롭: 드롭률>=0, 정상 감소\n- 정상_드롭_1주차부스트: 1주차 공휴일로 드롭 과대 가능\n- 공휴일_역주행: 2주차 공휴일이 원인 (외생적)\n- 순수_역주행: 공휴일 무관 (서울의 봄 타입)",
    '1주차_공휴일명'    : "### 1주차 공휴일명\n1주차 주말 전후(목~월)에 포함된 공휴일 이름/날짜\n\n- 비어있으면 해당 기간 공휴일 없음",
    '2주차_공휴일명'    : "### 2주차 공휴일명\n2주차 주말 전후(목~월)에 포함된 공휴일 이름/날짜\n\n- 공휴일 역주행 케이스 검증용",
    '1주차_주말일자'    : "### 1주차 주말일자\n1주차 주말 금~일 날짜 범위 (MM/DD~MM/DD)",
    '2주차_주말일자'    : "### 2주차 주말일자\n2주차 주말 금~일 날짜 범위 (MM/DD~MM/DD)",
    'count'             : "### count\n해당 그룹의 영화 개수",
    'mean'              : "### mean\n드롭률 평균 (%)",
    'median'            : "### median\n드롭률 중위수 (%)",
    'std'               : "### std\n드롭률 표준편차",
    'min'               : "### min\n드롭률 최솟값 (%)",
    'max'               : "### max\n드롭률 최댓값 (%)",
    '항목'              : "### 항목\n방법론 설명 항목명",
    '설명'              : "### 설명\n해당 항목의 정의 및 계산식",
    '출처'              : "### 출처\n데이터 원본 파일",
}

tier_colors = {'중형': 'FFE699', '중대형': 'F4B084', '대형': 'FFB6B6', '기타': 'D9D9D9'}
case_colors = {
    '정상_드롭'          : 'E2EFDA',
    '정상_드롭_1주차부스트': 'FFF2CC',
    '공휴일_역주행'      : 'FCE4D6',
    '순수_역주행'        : 'F4CCCC',
    '데이터_부족'        : 'D9D9D9',
}

wb = load_workbook(OUT_PATH)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # 헤더 서식 + 마크다운 메모 부착
    for cell in ws[1]:
        cell.font      = Font(name='맑은 고딕', bold=True, color='FFFFFF')
        cell.fill      = PatternFill('solid', start_color='305496')
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        desc = COLUMN_DESCRIPTIONS.get(str(cell.value))
        if desc:
            cmt        = Comment(desc, '데이터 사전')
            cmt.width  = 320
            cmt.height = max(120, 22 * (desc.count('\n') + 2))
            cell.comment = cmt

    # 본문 서식
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font      = Font(name='맑은 고딕', size=10)
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 컬럼 너비 자동 조정
    for col in ws.columns:
        try:
            max_len = max((len(str(c.value)) for c in col if c.value is not None), default=8)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 32)
        except Exception:
            pass

    # 체급·분류·드롭률 색상 코딩
    header   = {cell.value: cell.column for cell in ws[1]}
    tier_col = header.get('체급')
    case_col = header.get('분류')
    drop_col = header.get('드롭률(%)')

    for r in range(2, ws.max_row + 1):
        if tier_col:
            v = ws.cell(row=r, column=tier_col).value
            if v in tier_colors:
                ws.cell(row=r, column=tier_col).fill = PatternFill('solid', start_color=tier_colors[v])
                ws.cell(row=r, column=tier_col).font = Font(name='맑은 고딕', bold=True, size=10)
        if case_col:
            v = ws.cell(row=r, column=case_col).value
            if v in case_colors:
                ws.cell(row=r, column=case_col).fill = PatternFill('solid', start_color=case_colors[v])
                ws.cell(row=r, column=case_col).font = Font(name='맑은 고딕', bold=True, size=10)
        if drop_col:
            v = ws.cell(row=r, column=drop_col).value
            if isinstance(v, (int, float)) and v < 0:
                ws.cell(row=r, column=drop_col).font = Font(
                    name='맑은 고딕', bold=True, color='C00000', size=10)

    ws.freeze_panes = 'A2'

wb.save(OUT_PATH)
print(f"✓ 서식 적용 완료: {OUT_PATH}")

✓ 서식 적용 완료: /content/movie_드롭률_분석.xlsx


In [ ]:
files.download(OUT_PATH)
print("✓ 다운로드 시작 — 브라우저 다운로드 폴더를 확인하세요")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ 다운로드 시작 — 브라우저 다운로드 폴더를 확인하세요
